In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions.builtin import length

spark = SparkSession.builder\
    .appName("Spark aggregation function")\
    .getOrCreate()

In [ ]:
listings = spark.read.csv(
    "../data/listings.csv.gz",
    sep=",",
    quote='"',
    escape='"',
    header=True,
    inferSchema=True,
    multiLine=True,
    mode="PERMISSIVE"
)
listings.printSchema()

In [ ]:
reviews = spark.read.csv(
    "../data/reviews.csv.gz",
    sep=",",
    quote='"',
    escape='"',
    header=True,
    inferSchema=True,
    multiLine=True,
    mode="PERMISSIVE"
)
reviews.printSchema()

In [ ]:
# 1. Count the number of reviews per listing using the "reviews" dataset
import pyspark.sql.functions as F

reviews.groupBy(reviews.listing_id)\
    .agg(F.count('listing_id').alias('reviews_per_listing'))\
    .orderBy('reviews_per_listing', ascending=[False]).show()

In [10]:
# 2. Compute the total number of listings and average review score per host
listing_reviews = listings.join(
    reviews,
    listings.id == reviews.listing_id,
    how='inner'
)

listing_reviews.groupBy(listing_reviews.host_name)\
    .agg(
    F.count('host_name').alias('number_of_listings'),
    F.count(listings.id).alias('num_of_list_id'),
    F.avg('review_scores_rating').alias('avg_review_score')
).orderBy('number_of_listings', ascending=[False]).show(truncate=False)

+--------------+------------------+--------------+------------------+
|host_name     |number_of_listings|num_of_list_id|avg_review_score  |
+--------------+------------------+--------------+------------------+
|James         |21331             |21331         |4.715685621864779 |
|Alex          |17818             |17818         |4.717671455831241 |
|Sarah         |15659             |15659         |4.78941439427797  |
|David         |14688             |14688         |4.777833605664326 |
|Richard       |11491             |11491         |4.803409624923936 |
|Paul          |11485             |11485         |4.798588593818048 |
|Michael       |10305             |10305         |4.792306647258628 |
|Lisa          |10056             |10056         |4.868173229912598 |
|John          |10036             |10036         |4.785306895177296 |
|Mark          |9968              |9968          |4.749853531300197 |
|Anna          |9780              |9780          |4.806667689161839 |
|Ali           |9655

In [6]:
# 3. Find the top ten listings with the highest number of reviews
listing_reviews.groupBy(listings.name)\
    .agg(F.count(reviews.id).alias('count_of_reviews'))\
    .orderBy('count_of_reviews', ascending=[False])\
    .limit(10).show(truncate=False)

+--------------------------------------------------+----------------+
|name                                              |count_of_reviews|
+--------------------------------------------------+----------------+
|Double Room+ Ensuite                              |1902            |
|Private double room with en suite facilities      |1647            |
|Locke Studio Apartment at Leman Locke             |1443            |
|Double Garden View room - London House Hotel***   |1298            |
|Stunning Shoreditch High Street Studio            |1187            |
|Private Double Room in Warren Street              |1144            |
|London's best transport hub 5 mins walk! Safe too!|1142            |
|Superior Studio, avg size 23.5 msq                |1002            |
|Large Room + Private Bathroom, E3.                |998             |
|S - Heathrow Airport Terminal 2 3 4 5 Hatton Cross|951             |
+--------------------------------------------------+----------------+



In [13]:
# 3. Find the top ten listings with the highest number of reviews
reviews.printSchema()

reviews.groupBy('listing_id')\
    .count().orderBy('count', ascending=False)\
    .show()

root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)

+----------+-----+
|listing_id|count|
+----------+-----+
|  47408549| 1902|
|  43120947| 1647|
|  19670926| 1443|
|   2126708| 1142|
|  46233904| 1002|
|   2659707|  998|
|  27833488|  951|
|   4748665|  933|
|  42081759|  914|
|   5266466|  909|
|   2025844|  849|
|   4332039|  848|
|   4461052|  828|
|    107051|  796|
|   5869526|  791|
|  15988694|  789|
|   4972039|  743|
|  44204525|  732|
|     36660|  730|
|   4326511|  727|
+----------+-----+
only showing top 20 rows


In [7]:
# 4. Find the top five neighborhoods with the most listings
listing_reviews.groupBy(listing_reviews.host_neighbourhood)\
    .agg(F.count('host_neighbourhood').alias('nums_of_listings'))\
    .orderBy('nums_of_listings', ascending=[False])\
    .limit(5).show(truncate=False)

listing_reviews.printSchema()

# NO NO NO

+------------------+----------------+
|host_neighbourhood|nums_of_listings|
+------------------+----------------+
|Bayswater         |39154           |
|Pimlico           |37947           |
|Shoreditch        |32412           |
|Kensington        |28653           |
|Hammersmith       |27629           |
+------------------+----------------+

root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nu

In [17]:
listings.groupBy('neighbourhood_cleansed')\
    .count().orderBy('count', ascending=False)\
    .limit(5).show(truncate=False)

+----------------------+-----+
|neighbourhood_cleansed|count|
+----------------------+-----+
|Westminster           |11385|
|Tower Hamlets         |7469 |
|Camden                |6551 |
|Kensington and Chelsea|6401 |
|Hackney               |6359 |
+----------------------+-----+



In [8]:
# 5. Get a data frame with the following four columns:
# Listing's ID
# Listing's name
# Reviewer's name
# Review's comment
# Use "join" to combine data from two datasets

listing_reviews.select(listings.id,
                        listing_reviews.name,
                        listing_reviews.reviewer_name,
                        listing_reviews.comments)\
    .show(20)

+-----+--------------------+-------------+--------------------+
|   id|                name|reviewer_name|            comments|
+-----+--------------------+-------------+--------------------+
|13913|Holiday London DB...|      Michael|My girlfriend and...|
|13913|Holiday London DB...|      Mathias|Alina was a reall...|
|13913|Holiday London DB...|      Kristin|Alina is an amazi...|
|13913|Holiday London DB...|      Camilla|Alina's place is ...|
|13913|Holiday London DB...|        Jorik|Nice location in ...|
|13913|Holiday London DB...|         Vera|I'm very happy to...|
|13913|Holiday London DB...|         Honi|I stayed with Ali...|
|13913|Holiday London DB...|   Alessandro|Alina was a perfe...|
|13913|Holiday London DB...|         Oleh|Alina's flat is e...|
|13913|Holiday London DB...|           Mo|The House is a pi...|
|13913|Holiday London DB...|            A|Was great base fo...|
|13913|Holiday London DB...|       Daniel|Alina was an amaz...|
|13913|Holiday London DB...|      Belind

In [39]:
# 6. Get top five listings with the highest average review comment length. Only return listings with at least 5 reviews
from pyspark.sql.functions import length, avg, count

review_with_comments_length = reviews.withColumn('comment_length', length('comments'))

reviews.printSchema()
review_with_comments_length.show(truncate=True)

review_with_comments_length\
    .join(listings, review_with_comments_length.listing_id == listings.id, 'inner')\
    .groupBy('listing_id').agg(F.avg('comment_length').alias('average_comment_length'),
                               F.count(review_with_comments_length.id).alias('number_of_reviews'))\
    .filter('number_of_reviews >= 5').orderBy('average_comment_length', ascending=False)\
    .show()

root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)

+----------+---------+----------+-----------+-------------+--------------------+--------------+
|listing_id|       id|      date|reviewer_id|reviewer_name|            comments|comment_length|
+----------+---------+----------+-----------+-------------+--------------------+--------------+
|     13913|    80770|2010-08-18|     177109|      Michael|My girlfriend and...|           873|
|     13913|   367568|2011-07-11|   19835707|      Mathias|Alina was a reall...|           172|
|     13913|   529579|2011-09-13|    1110304|      Kristin|Alina is an amazi...|           350|
|     13913|   595481|2011-10-03|    1216358|      Camilla|Alina's place is ...|           409|
|     13913|   612947|2011-10-09|     490840|        Jorik|Nice location in ...|       

In [44]:
# 7. Using the "join" operator find listings without reviews.
# Hint: Use "left_join" or "left_anti" join type when implementing this
# Performing a left anti join to find listings without reviews

joined_df = listings.join(reviews, listings.id == reviews.listing_id, 'left_outer')

joined_df.filter(reviews.id.isNull()).select('name').show(truncate=False)

listings.join(reviews, listings.id == reviews.listing_id, 'left_anti')\
    .select('name').show(truncate=False)

+-------------------------------------------------+
|name                                             |
+-------------------------------------------------+
|ChiqDoube Room in PrivateAppartment              |
|ROOM TO RENT IN THE OLYMPIC PERIOD               |
|Luxury Central London House with Gym and Reformer|
|Stunning Shared Penthouse Apartment              |
|SPARE ROOM TO LET DURING OLYMPICS                |
|Your Studio Flat in London-Olympics              |
|Large Cosy Apartment with Garden E7              |
|Central London flat + skyline views              |
|Trellick Tower, Portobello Rd.                   |
|AROOM TO LET - month                             |
|Cosy 1 bedroom with Thames View                  |
|Single bedroom or double bedroom                 |
|Beauitiful Mordern House Ensuite Spacious        |
|Luxury 1 bedroom apartment                       |
|Double Bedroom Hackney                           |
|Gorgeous Sunny Bedroom in Brixton                |
|Renting: 28